# Instalacija i uvoz potrebnih biblioteka

In [ ]:
%pip install -r requirements.txt

In [ ]:
import cv2
import mediapipe as mp
import tensorflow as tf

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp

import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

# Pomoćne funkcije

In [ ]:
# Initializes the hands model from the MediaPipe library used for detecting and tracking hand landmarks.
mp_hands = mp.solutions.hands
# Initializes the drawing utilities from the MediaPipe library used to draw the detected landmarks on the image.
mp_drawing = mp.solutions.drawing_utils

def mediapipe_detection(image, model):
    '''
    Processes an image using a MediaPipe model to detect landmarks.
    Args:
        image: The image to process.
        model: The MediaPipe model to use for detection.
    Returns:
        image: The image with the detected landmarks.
        results: The landmarks detected in the image.
    '''
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

def draw_styled_landmarks(image, results):
    '''
    Draws the detected landmarks on the image.

    Args: 
        image: The image to draw the landmarks on.
        results: The landmarks detected in the image.

    Returns:
        image: The image with the detected landmarks drawn on it.
    '''
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS) # Draw hand connections

def extract_keypoints(results):
    '''
    Extracts the keypoints from the results of the MediaPipe model.

    Args:
        results: The results of the MediaPipe model

    Returns:
        np.array: The keypoints extracted from the results of the MediaPipe model
    '''
    if results.multi_hand_landmarks:
        lh = np.array([[res.x, res.y, res.z] for res in results.multi_hand_landmarks[0].landmark]).flatten() if len(results.multi_hand_landmarks) > 0 else np.zeros(21*3)
        rh = np.array([[res.x, res.y, res.z] for res in results.multi_hand_landmarks[1].landmark]).flatten() if len(results.multi_hand_landmarks) > 1 else np.zeros(21*3)
    else:
        lh = np.zeros(21*3)
        rh = np.zeros(21*3)
    return np.concatenate([lh, rh])

def prob_viz(res, actions, input_frame, colors):
    '''
    Visualizes the probabilities of the actions detected.
    
    Args:
        res: The probabilities of the actions detected.
        actions: The actions detected.
        input_frame: The frame to draw the visualization on.
        colors: The colors to use for the visualization.
        
    Returns:
        output_frame: The frame with the visualization drawn on it.
    '''
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)
        
    return output_frame

# Priprema foldera

In [ ]:
# amount of data used
# 3 videos worth of data
no_sequences = 3

# Videos are going to be 30 frames in length
sequence_length = 5


In [ ]:
# Path for exported data, numpy arrays
DATA_PATH = os.path.join('Data_2')

# actions that we are detecting
actions = np.array(['hello', 'thanks', 'iloveyou'])


# creating folders
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            # already exists
            pass

# Prikupljanje ključnih tačaka za treniranje i testiranje


In [ ]:
# Collecting Keypoints for training and testing
cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    for action in actions:
        for sequence in range(no_sequences):
            for frame_num in range(sequence_length):

                countdown_ms = 1000   # 1000 = 1 s, stavi 500 za 0.5 s
                step = 100

                # Countdown
                for remaining in range(countdown_ms, 0, -step):

                    ret, frame = cap.read()
                    if not ret:
                        continue

                    image, results = mediapipe_detection(frame, hands)
                    draw_styled_landmarks(image, results)

                    seconds = remaining / 1000

                    cv2.putText(
                        image,
                        f"Gesture: {action}",
                        (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (255, 255, 255),
                        2
                    )

                    cv2.putText(
                        image,
                        f"Video: {sequence+1}/{no_sequences}",
                        (20, 75),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (255, 255, 255),
                        2
                    )

                    cv2.putText(
                        image,
                        f"Frame: {frame_num+1}/{sequence_length}",
                        (20, 110),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (255, 255, 255),
                        2
                    )

                    cv2.putText(
                        image,
                        f"Capturing in {seconds:.1f}s",
                        (120, 240),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0, 255, 255),
                        3
                    )

                    cv2.imshow("OpenCV Feed", image)

                    if cv2.waitKey(step) & 0xFF == ord('q'):
                        cap.release()
                        cv2.destroyAllWindows()
                        raise SystemExit

                # Capture frame after countdown
                ret, frame = cap.read()
                if not ret:
                    continue

                image, results = mediapipe_detection(frame, hands)
                draw_styled_landmarks(image, results)

                keypoints = extract_keypoints(results)

                npy_path = os.path.join(
                    DATA_PATH,
                    action,
                    str(sequence),
                    str(frame_num)
                )

                np.save(npy_path, keypoints)

                cv2.putText(
                    image,
                    "Captured!",
                    (180, 240),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    3
                )

                cv2.imshow("OpenCV Feed", image)
                cv2.waitKey(200)

cap.release()
cv2.destroyAllWindows()

# Priprema podataka


In [ ]:
label_map = {label:num for num, label in enumerate(actions)}

# load and process gesture data from data path
sequences, labels = [], []
for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        # loading frames from each sequence
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), f'{frame_num}.npy'))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])



X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=0,
    stratify=np.argmax(y, axis=1)
)

# Treniranje LSTM modela

In [76]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, activation='relu', input_shape=input_shape),
        LSTM(128, return_sequences=True, activation='relu'),
        LSTM(64, return_sequences=False, activation='relu'),
        Dense(64, activation='relu'),
        Dense(32, activation='relu'),
        Dense(output_shape, activation='softmax')
    ]
)

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
# model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor='loss', patience=10)
model.fit(X_train, y_train, epochs=1000, callbacks=[early_stopping])

model.save('gesture_model.h5')

# Evaluacija modela


In [ ]:
# Test the model
y_pred = model.predict(X_test)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true, y_pred, labels=range(len(actions)))
cm_matrix = pd.DataFrame(cm, index=actions, columns=actions)

sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="viridis")

print(classification_report(
    y_true,
    y_pred,
    target_names=actions
))

# Prepoznavanje gesti u stvarnom vremenu

In [ ]:
model = load_model('gesture_model.h5')

# Used for visualization
colors = [(245,117,16), (117,245,16), (16,117,245)]

# Detection variables
sequence = []
display_text = ""
threshold = 0.5

cap = cv2.VideoCapture(0)

# Set MediaPipe model
with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():

        # Read frame
        ret, frame = cap.read()
        if not ret:
            break

        # Make detections
        image, results = mediapipe_detection(frame, hands)

        # Draw landmarks
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-sequence_length:]

        if len(sequence) == sequence_length:

            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]

            if res[np.argmax(res)] > threshold:
                display_text = actions[np.argmax(res)]
            else:
                display_text = "Unknown"

            # Draw probability bars
            image = prob_viz(res, actions, image, colors)

        # Display detected gesture
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)

        cv2.putText(
            image,
            display_text,
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        # Show image
        cv2.imshow("OpenCV Feed", image)

        # Exit on 'q'
        if cv2.waitKey(100) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()